In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 可选单候选可读性诊断，B优先

复用现有3-cell运行骨架，从A开发分支获取已审方法版本。默认执行一次固定候选诊断：3次生成、4张最终图、clean/+10°共12条完整盲路径，另有2次无anchor reader诊断。无新模板、搜索或候选扩展。设EXECUTE=False仅打印CPU计划。

输出保存到Drive的CEG-WM/development/latent-sync-v1/diagnostic-时间目录。需要Colab GPU及Secrets中的HF_TOKEN、CEG_WM_ROOT_KEY；环境版本仅记录。检查report.json、rows.jsonl及保存图像；失败结果保留。此候选仍未经真实模型验证，不代表潜空间同步成功或创新成立。若clean仍不可读或仅私有参考可读，停止本候选。


In [ ]:
import json, os, pathlib, subprocess, sys, datetime
from importlib.metadata import version, PackageNotFoundError

REPO='https://github.com/RICHAAARC/CEG-WM.git'
BRANCH='dev/latent-sync-v1'
EXPECTED_EXACT='8a644206ba042abfdfda233a4e776816f95457ae'
checkout=pathlib.Path('/content/latent-sync-v1-github')
runtime_root=pathlib.Path('/content/ceg-method-models')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/development/latent-sync-v1')
EXECUTE=True  # False: CPU plan only
UNIT_INDEX=0
output_root=drive_root/('diagnostic-'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
if not checkout.exists(): subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'fetch','origin',BRANCH],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate','lpips','torchmetrics','scipy','sentencepiece'],check=True)
environment={}
for package in ('torch','diffusers','transformers','accelerate'):
    try: environment[package]=version(package)
    except PackageNotFoundError: environment[package]=None
print({'branch':BRANCH,'code':EXPECTED_EXACT,'versions':environment})
child_env=dict(os.environ)
child_env['PYTHONPATH']=str(checkout/'src')+os.pathsep+str(checkout)
if EXECUTE:
    from google.colab import userdata
    child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
    child_env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY') or ''
unit=json.loads((checkout/'configs/parallel_method_dev/fit.json').read_text())[UNIT_INDEX]
command=[sys.executable,'-m','experiments.run_latent_sync_development',
         '--output',str(output_root),'--seed',str(unit['seed']),'--prompt',unit['prompt']]
if EXECUTE: command += ['--execute']
completed=subprocess.run(command,cwd=checkout,env=child_env,check=False)
print('输出目录:', output_root)
if completed.returncode != 0:
    raise RuntimeError(f'开发入口退出码 {completed.returncode}；已写入的结果与失败行保留在 {output_root}')
report_path=output_root/'report.json'
if report_path.exists():
    report=json.loads(report_path.read_text())
    print({'report':str(report_path),'row_errors':report.get('row_errors'),'claim':report.get('claim')})
